<a href="https://colab.research.google.com/github/l-isaro/Activity_image-processing/blob/main/air_quality_lstm_reboot_patched.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Patched Notebook — Baseline LSTM (WINDOW=60, No Rolling)
This version implements:
- Baseline features only (no rolling means)
- Safe time split + scaling
- WINDOW = 60 with rebuilt sequences for train/val/test
- LSTM(128→64), Dropout 0.15, Adam 1e-3
- Eval + Submit prints Validation RMSE and saves Kaggle-formatted CSV


In [ ]:

# Imports & setup
import numpy as np, pandas as pd
from pathlib import Path
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from math import sqrt

seed = 42
np.random.seed(seed)
tf.random.set_seed(seed)

DATA_DIR = Path('data')
TRAIN_PATH = DATA_DIR / 'train.csv'
TEST_PATH  = DATA_DIR / 'test.csv'

train = pd.read_csv(TRAIN_PATH, parse_dates=['datetime'])
test  = pd.read_csv(TEST_PATH,  parse_dates=['datetime'])
train.shape, test.shape


((30676, 12), (13148, 11))

In [ ]:

# ===== FEATURE ENGINEERING (baseline only; no rolling) =====
TARGET = 'pm2.5'
DROP_COLS = ['No']

df = train.dropna(subset=[TARGET]).copy()
test = test.copy()
for c in DROP_COLS:
    if c in df.columns: df.drop(columns=c, inplace=True)
    if c in test.columns: test.drop(columns=c, inplace=True)

df['datetime'] = pd.to_datetime(df['datetime']); test['datetime'] = pd.to_datetime(test['datetime'])
df = df.sort_values('datetime').reset_index(drop=True)
test = test.sort_values('datetime').reset_index(drop=True)

def add_time_features(frame):
    f = frame.copy()
    f['hour'] = f['datetime'].dt.hour
    f['dayofweek'] = f['datetime'].dt.dayofweek
    f['month'] = f['datetime'].dt.month
    f['hour_sin'] = np.sin(2*np.pi*f['hour']/24)
    f['hour_cos'] = np.cos(2*np.pi*f['hour']/24)
    f['dow_sin']  = np.sin(2*np.pi*f['dayofweek']/7)
    f['dow_cos']  = np.cos(2*np.pi*f['dayofweek']/7)
    f['mon_sin']  = np.sin(2*np.pi*(f['month']-1)/12)
    f['mon_cos']  = np.cos(2*np.pi*(f['month']-1)/12)
    return f

df = add_time_features(df); test = add_time_features(test)

FEATURES = [c for c in df.columns if c not in ['pm2.5','datetime'] and not c.endswith(('_ma6','_ma12','_ma24'))]
len(FEATURES), FEATURES[:10]


(18,
 ['DEWP',
  'TEMP',
  'PRES',
  'Iws',
  'Is',
  'Ir',
  'cbwd_NW',
  'cbwd_SE',
  'cbwd_cv',
  'hour'])

In [ ]:

# ===== SAFE TIME SPLIT + SCALING =====
fixed_cut = pd.Timestamp('2013-01-01 00:00:00')
if df['datetime'].min() < fixed_cut < df['datetime'].max():
    VAL_START = fixed_cut
else:
    VAL_START = df['datetime'].quantile(0.85)

train_df = df[df['datetime'] < VAL_START].copy()
val_df   = df[df['datetime'] >= VAL_START].copy()

print("Split at:", VAL_START, "| Train rows:", len(train_df), "Val rows:", len(val_df))
print("Date range:", df['datetime'].min(), "→", df['datetime'].max())

train_medians = train_df[FEATURES].median()
train_df[FEATURES] = train_df[FEATURES].fillna(train_medians)
val_df[FEATURES]   = val_df[FEATURES].fillna(train_medians)

scaler = StandardScaler()
train_X = scaler.fit_transform(train_df[FEATURES].values)
val_X   = scaler.transform(val_df[FEATURES].values)

train_y = train_df[TARGET].values
val_y   = val_df[TARGET].values
train_time = train_df['datetime'].values
val_time   = val_df['datetime'].values

train_X.shape, val_X.shape


Split at: 2013-01-01 00:00:00 | Train rows: 24418 Val rows: 4337
Date range: 2010-01-02 00:00:00 → 2013-07-02 03:00:00


((24418, 18), (4337, 18))

In [ ]:

# ===== SEQUENCES (WINDOW=60) =====
WINDOW = 60
HORIZON = 0
print("Using WINDOW =", WINDOW)

def make_sequences(X, y, window=60, horizon=0):
    Xs, ys = [], []
    for i in range(window, len(X) - horizon):
        Xs.append(X[i-window:i, :])
        ys.append(y[i + horizon])
    return np.array(Xs), np.array(ys)

X_tr, y_tr = make_sequences(train_X, train_y, window=WINDOW, horizon=HORIZON)
X_va, y_va = make_sequences(val_X,   val_y,   window=WINDOW, horizon=HORIZON)
X_tr.shape, X_va.shape


Using WINDOW = 60


((24358, 60, 18), (4277, 60, 18))

In [ ]:

# ===== MODEL (LSTM 128→64, Dropout 0.15) =====
inputs = keras.Input(shape=(X_tr.shape[1], X_tr.shape[2]))
x = layers.Masking(mask_value=0.0)(inputs)
x = layers.LSTM(128, return_sequences=True)(x)
x = layers.Dropout(0.15)(x)
x = layers.LSTM(64)(x)
x = layers.Dropout(0.15)(x)
x = layers.Dense(64, activation='relu')(x)
outputs = layers.Dense(1)(x)

model = keras.Model(inputs, outputs)
model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss='mse')

callbacks = [
    keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor='val_loss'),
    keras.callbacks.ReduceLROnPlateau(patience=3, factor=0.5, verbose=1),
]

EPOCHS = 50
BATCH_SIZE = 256
history = model.fit(
    X_tr, y_tr,
    validation_data=(X_va, y_va),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=2,
    callbacks=callbacks
)

val_pred = model.predict(X_va, verbose=0).squeeze()
val_rmse = sqrt(mean_squared_error(y_va, val_pred))
print(f"\nValidation RMSE: {val_rmse}")


Epoch 1/50
96/96 - 59s - 613ms/step - loss: 13534.6963 - val_loss: 15690.1865 - learning_rate: 1.0000e-03
Epoch 2/50
96/96 - 83s - 862ms/step - loss: 8244.3662 - val_loss: 12828.2939 - learning_rate: 1.0000e-03
Epoch 3/50
96/96 - 52s - 543ms/step - loss: 8032.1167 - val_loss: 12974.7051 - learning_rate: 1.0000e-03
Epoch 4/50
96/96 - 84s - 875ms/step - loss: 6610.6050 - val_loss: 9640.6719 - learning_rate: 1.0000e-03
Epoch 5/50
96/96 - 54s - 561ms/step - loss: 5498.4346 - val_loss: 8546.0439 - learning_rate: 1.0000e-03
Epoch 6/50
96/96 - 53s - 554ms/step - loss: 4794.9219 - val_loss: 7583.6265 - learning_rate: 1.0000e-03
Epoch 7/50
96/96 - 53s - 552ms/step - loss: 4299.4648 - val_loss: 7113.4673 - learning_rate: 1.0000e-03
Epoch 8/50
96/96 - 81s - 845ms/step - loss: 4108.3945 - val_loss: 7542.9707 - learning_rate: 1.0000e-03
Epoch 9/50
96/96 - 55s - 568ms/step - loss: 3904.8816 - val_loss: 6917.6250 - learning_rate: 1.0000e-03
Epoch 10/50
96/96 - 82s - 856ms/step - loss: 3615.1616 - val

In [ ]:

# ===== TEST SEQUENCES (match WINDOW) =====
full = pd.concat([df[['datetime'] + FEATURES], test[['datetime'] + FEATURES]], ignore_index=True).sort_values('datetime')
full_X = scaler.transform(full[FEATURES].values)
idx_map = {t: i for i, t in enumerate(full['datetime'].values)}

def make_test_sequences(full_X, full_time, end_times, window=60):
    Xs, valid_times = [], []
    for t in end_times:
        i = idx_map.get(t, None)
        if i is None or i < window:
            continue
        Xs.append(full_X[i-window:i, :])
        valid_times.append(t)
    return np.array(Xs), np.array(valid_times)

X_te, te_times = make_test_sequences(full_X, full['datetime'].values, test['datetime'].values, window=WINDOW)
X_te.shape, len(te_times)


((13148, 60, 18), 13148)

In [ ]:

# ===== EVAL + SUBMIT =====
# Validation RMSE
val_pred = model.predict(X_va, verbose=0).squeeze()
val_rmse = sqrt(mean_squared_error(y_va, val_pred))
print(f"\nValidation RMSE: {val_rmse}")

# Test predictions aligned to EXACT test order
test_pred = model.predict(X_te, verbose=0).squeeze()
pred_series = pd.Series(test_pred, index=pd.to_datetime(te_times))
ordered_pred = pred_series.reindex(pd.to_datetime(test['datetime'])).values

# light clipping
ordered_pred = np.clip(ordered_pred, 0, 1000)

# exact Kaggle format
try:
    row_id = test["datetime"].dt.strftime("%Y-%m-%d %-H:00:00")
except Exception:
    row_id = test["datetime"].dt.strftime("%Y-%m-%d %#H:00:00")

submission = pd.DataFrame({"row ID": row_id, "pm2.5": ordered_pred})
out_path = Path("data/submission_window60_baseline.csv")
submission.to_csv(out_path, index=False)
print("Saved:", out_path, submission.shape)
submission.head()



Validation RMSE: 78.47603640649068
Saved: data/submission_window60_baseline.csv (13148, 2)


,row ID,pm2.5
0,2013-07-02 4:00:00,47.119919
1,2013-07-02 5:00:00,46.320778
2,2013-07-02 6:00:00,44.637394
3,2013-07-02 7:00:00,43.321777
4,2013-07-02 8:00:00,41.710686
